# Benchmark 1/3 — Pitch Detection

Component test for the **pYIN front-end + PitchSmoother (HMM stage 2)** against
frame-level f0 ground truth, scored with `mir_eval.melody`.

All logic lives in `PitchBenchmarker` (see `benchmarks/PitchBenchmarker.py`); the pipeline
runs detection **and** smoothing, and **auto-tightens [fmin, fmax] to each track**
from its annotation (PitchDetector pads ±0.5 semitone internally, so the exact
min/max go into the Config).

**Datasets** (`PitchBenchmarker.PITCH_DATASETS`): monophonic, single-f0 only —
`mdb-stem-synth`, `mdb-melody-synth`, `bach10-mf0-synth`. `mdb-mf0-synth` is
excluded (polyphonic mix).

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import sys
sys.path.insert(0, "../benchmarks")
from PitchBenchmarker import PitchBenchmarker
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
bm = PitchBenchmarker(max_tracks=15)   # cap per dataset; None = all

## Quick run — `mdb-stem-synth`

In [ ]:
df = bm.bench_pitch_dataset("mdb-stem-synth", max_tracks=8, write=True)
bm.summarize(df, cols=bm.PITCH_METRICS, name="mdb-stem-synth (smoothed + tightened)")
df[bm.PITCH_METRICS]

## Ablation — does the smoother + range-tightening help?
Four corners of (raw vs smoothed) × (wide vs tightened range).

In [ ]:
ablation = pd.DataFrame({
    f"{'smoothed' if s else 'raw'} + {'tightened' if t else 'wide'}":
        bm.bench_pitch_dataset("mdb-stem-synth", max_tracks=8, smooth=s,
                               tighten_pitch_range=t, verbose=False)[bm.PITCH_METRICS].mean()
    for s in (False, True) for t in (False, True)
}).T
ablation

## Full run across datasets (saves CSVs to `benchmarks/results/pitch/`)

In [ ]:
summary = {}
for ds in bm.PITCH_DATASETS:
    df = bm.bench_pitch_dataset(ds, verbose=False, write=True)
    summary[ds] = df[bm.PITCH_METRICS].mean()
    print(f"{ds:18s} {len(df):>3} tracks")
pd.DataFrame(summary).T